# News Classification (Simplified, Standalone)

This notebook is the **main operational notebook** for `Code/News-Classification-NLP`.

It is aligned with the new standalone project design:
- News-only dataset (`fake.csv`, `true.csv`)
- Hyperparameter tuning pipeline via scripts
- No dependency on IMDB or `Code/datasets/processed`


## How this notebook works

Instead of duplicating training logic in many cells, this notebook calls the project scripts directly:
1. `data_loader.py`
2. `run_preprocessing.py`
3. `train_baseline_models.py`
4. `train_advanced_models.py`
5. `evaluate.py`

This keeps the notebook simple, reproducible, and synchronized with production scripts.


In [ ]:
from pathlib import Path
import sys
import subprocess
import json
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Resolve project directory robustly
cwd = Path.cwd().resolve()
if (cwd / "Code" / "News-Classification-NLP").exists():
    PROJECT_DIR = (cwd / "Code" / "News-Classification-NLP").resolve()
elif (cwd / "run_preprocessing.py").exists() and (cwd / "app.py").exists():
    PROJECT_DIR = cwd
else:
    raise RuntimeError("Cannot locate News-Classification-NLP folder from current working directory.")

DATA_RAW = PROJECT_DIR / "data" / "raw"
DATA_PROCESSED = PROJECT_DIR / "data" / "processed"
MODELS_DIR = PROJECT_DIR / "models"
REPORTS_DIR = PROJECT_DIR / "reports"
PLOTS_DIR = PROJECT_DIR / "evaluation_plots"

print("PROJECT_DIR:", PROJECT_DIR)
print("Python:", sys.executable)


In [ ]:
def run_cmd(*args):
    cmd = [sys.executable, *args]
    print("\n$", " ".join(str(x) for x in cmd))
    result = subprocess.run(cmd, cwd=PROJECT_DIR, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result


## 1) Dataset check / auto-download

This uses Kaggle if `fake.csv` / `true.csv` are missing.

If Kaggle credentials are not configured, manually place:
- `data/raw/fake.csv`
- `data/raw/true.csv`


In [ ]:
run_cmd(
    "data_loader.py",
    "--data-dir", "data/raw",
    "--download-if-missing"
)


## 2) Preprocessing

In [ ]:
run_cmd(
    "run_preprocessing.py",
    "--data-dir", "data/raw",
    "--out-dir", "data/processed"
)


## 3) Baseline hyperparameter tuning (NB + Logistic Regression)

In [ ]:
run_cmd(
    "train_baseline_models.py",
    "--processed-dir", "data/processed"
)


## 4) Advanced hyperparameter tuning (LinearSVC + RandomForest)

In [ ]:
run_cmd(
    "train_advanced_models.py",
    "--processed-dir", "data/processed"
)


## 5) Evaluation report and plots

In [ ]:
run_cmd(
    "evaluate.py",
    "--models-dir", "models",
    "--processed-dir", "data/processed"
)


## 6) Read evaluation summary

In [ ]:
with open(REPORTS_DIR / "evaluation_report.json", "r") as fh:
    eval_report = json.load(fh)

print("Best model:", eval_report["best_model"]["name"])
pd.DataFrame(eval_report["models"]).T


## 7) Show generated plots

In [ ]:
plot_files = [
    PLOTS_DIR / "confusion_matrices.png",
    PLOTS_DIR / "roc_curves.png",
    PLOTS_DIR / "model_comparison.png",
]

for plot_path in plot_files:
    if plot_path.exists():
        img = plt.imread(plot_path)
        plt.figure(figsize=(12, 6))
        plt.imshow(img)
        plt.axis("off")
        plt.title(plot_path.name)
        plt.show()
    else:
        print("Missing plot:", plot_path)


## 8) Quick prediction test from best model bundle

In [ ]:
bundle_path = MODELS_DIR / "best_model_bundle.pkl"
with open(bundle_path, "rb") as fh:
    bundle = pickle.load(fh)

model = bundle["model"]
vectorizer = bundle["vectorizer"]
preprocessor = bundle["preprocessor"]
label_map = bundle["label_map"]

def predict_text(text):
    clean = preprocessor.preprocess(text)
    X = vectorizer.transform([clean])
    pred = int(model.predict(X)[0])
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)[0]
        conf = float(p[pred])
        real_score = float(p[1])
    elif hasattr(model, "decision_function"):
        raw = float(model.decision_function(X)[0])
        real_score = float(1 / (1 + np.exp(-raw)))
        conf = real_score if pred == 1 else (1 - real_score)
    else:
        real_score = None
        conf = 0.0

    return {
        "label": label_map.get(pred, "Real News" if pred == 1 else "Fake News"),
        "confidence": conf,
        "real_score": real_score,
    }

fake_example = "BREAKING: Government secretly installing mind control chips in vaccines activated by 5G towers."
real_example = "French lawmakers voted to oust Prime Minister François Bayrou Monday, plunging the country into a new political crisis."

print("Fake example ->", predict_text(fake_example))
print("Real example ->", predict_text(real_example))


## 9) Optional: run Streamlit app

Run this in terminal:

```bash
cd /Users/weichn/Applications/smartgit/nlp/Code/News-Classification-NLP
streamlit run app.py
```
